## Export, Process and Upload E*Trade Equity/Bond Portfolios 

1.  Login to E*Trade Porfolios Postions
    https://us.etrade.com/etx/pxy/portfolios/positions?_formtarget=portfoliolist

2.  Select each Account in turn from the dropdown list box.

3.  Select **Bonds** and/or **equity view** from the View dropdown

4.  On each view, click the **Download** icon to download the 'PortfolioDownload (n).csv' file

5.  Manually move the portfolio download CSV files to the 'local_folder' designated below.

6.  Run the following cells in this notebook to convert the CSV to JSON and upload to S3.  

7.  An S3 ObjectCreated event will initiate loading the JSON into DynamoDB.  

8.  Use the [excel_pipeline.ipynb](excel_pipeline.ipynb) refresh the appropriate etrade portfolio workbooks

![screenshot](EtradePortfolio.png)



In [ ]:
# Process and Upload E*Trade Equity/Bond Portfolios
import datetime
import os
import dotenv
dotenv.load_dotenv()

now = datetime.datetime.now()
yymmdd = now.strftime("%y%m%d" ) 
today_str = datetime.date.today().strftime("%Y-%m-%d")
# Get folder date from user input with today_str as default
today_str = input(f"Folder Date [{today_str}]: ") or today_str

s3_bucket_name = "fh-danelfin-289755104220"
s3_folder = f"inbound/account_data"
local_folder=f"/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/ingest/account_data/{today_str}"

if not os.getenv("S3_BUCKET_NAME"):
    os.environ["S3_BUCKET_NAME"] = s3_bucket_name
if not os.getenv("S3_FOLDER"):
    os.environ["S3_FOLDER"] = s3_folder
if not os.getenv("LOCAL_FOLDER"):
    os.environ["LOCAL_FOLDER"] = local_folder

print("Defaults:")
print("S3_BUCKET_NAME:", s3_bucket_name)
print("S3_FOLDER:", s3_folder)
print("LOCAL_FOLDER:", local_folder)

print(f"\nManually move downloaded E*TRADE CSV files to local folder: \n{local_folder}")
print(f"\nConverted JSON files will be uploaded to S3 folder: \ns3://{s3_bucket_name}/{s3_folder}\n")

print("Press Enter to continue...")
input("Press Enter to continue...")

from etrade_process_csv import process_etrade_csv_downloads

df_concat_stocks, df_concat_bonds = process_etrade_csv_downloads(local_folder, outpath=local_folder) #, yymmdd=yymmdd)
#df_concat_bonds.head(),df_concat_stocks.head()
print("CSV Upload and Processing complete.  Use 'excel_pipeline.ipynb' to refresh Excel")


Defaults:
S3_BUCKET_NAME: fh-danelfin-289755104220
S3_FOLDER: inbound/account_data
LOCAL_FOLDER: /mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/ingest/account_data/2026-11-02

Manually move downloaded E*TRADE CSV files to local folder: 
/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/ingest/account_data/2026-11-02

Converted JSON files will be uploaded to S3 folder: 
s3://fh-danelfin-289755104220/inbound/account_data

Press Enter to continue...
Processing /mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/ingest/account_data/2026-11-02/PortfolioDownload (35).csv, Account RAM BRKG -3543
Saved and uploaded /mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/ingest/account_data/2026-11-02/RAM-BRKG--3543.etrade.json to s3://fh-danelfin-289755104220/inbound/account_data/RAM-BRKG--3543.etrade.json
Processing /mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/ingest/account_data/2026-11-02/PortfolioDownload (36).csv, Account RAM ROLL -1969
Saved and uploaded /mnt/c/Users/richa

## Export/Upload/Refresh Workflow Complete
Sandbox follows ...

## Refresh Excel Pipeline (moved to excel_pipeline.ipynb)

8.  After files are uploaded to S3, execute the following see to download refreshed data to local CSV files.

9.  Open [etrade_portfolio.xlsx](/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/etrade_portfolio.xlsx) to access local copy of refreshed data.


In [ ]:
"""### moved to excel_pipeline.ipynb
from datetime import datetime
today_str = datetime.now().strftime("%Y-%m-%d")
win_outpath = "/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/excel_pipeline/current"
local_outpath = win_outpath

print(f"Today's date prefix is: {today_str}")
print(f"Windows output path is: {win_outpath}")

%reload_ext autoreload
%autoreload 2

from ramlytics_database import DynamoDBClient
from ramlytics_database import DbQuery
from ramlytics_yfinance import yf_search

qry = DbQuery(DynamoDBClient())

def store_csv(outpath, df, filename):
    import os
    from datetime import datetime
    today_str = datetime.today().strftime('%Y-%m-%d')

    df.to_csv(f"{outpath}/{filename}", index=False)
    # Save a copy of the CSV in the archive folder with today's date
    archive = outpath.replace("current", f"archive/{today_str}")
    os.makedirs(archive, exist_ok=True)
    df.to_csv(f"{archive}/{filename}", index=False)

    return f"{outpath}/{filename}", f"{archive}/{filename}"

def refresh_instruments(symbols = []):
    if not symbols:
        symbols = qry.instruments_df['symbol'].tolist()
    for symbol in symbols:
        inst = qry.db.get_instrument(symbol)
        inst.pop("symbol")
        response = qry.db.put_instrument(symbol, **inst)  

    qry.get_instruments_df()
      
    return f"refreshed symbols: {symbols}, response: {response}"

def refresh_all_csv(outpath="."):
    refresh_instruments()
    qry = DbQuery(DynamoDBClient())    

    store_csv(outpath, df=qry.get_instrument_market_summary_df(), filename="market_summary.csv")
    store_csv(outpath, df=qry.get_account_headers_df(), filename="account_headers.csv")
    store_csv(outpath, df=qry.get_acct_summary_by_symbol_df(), filename="acct_summary_by_symbol.csv")
    store_csv(outpath, df=qry.get_instrument_holdings_df(), filename="instrument_holdings.csv")
    store_csv(outpath, df=qry.get_instrument_scores_df(), filename="instrument_scores.csv")
    store_csv(outpath, df=qry.get_account_positions_df(), filename="account_positions.csv")

    return

refresh_all_csv(outpath=win_outpath)

print("CSV refresh complete.")
print(f"Files stored in: {win_outpath}")"""

Hello, world!
Today's date prefix is: 2026-11-02
Windows output path is: /mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/excel_pipeline/current
CSV refresh complete.
Files stored in: /mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/excel_pipeline/current


In [ ]:
df_concat_stocks

In [ ]:
today_str = input(f"Folder Date [{today_str}]: ") or today_str
today_str

In [3]:
# Export account summary by symbol to Yahoo format
# rename columns for yahoo finance
# e.g. 
# Symbol	Trade Date	Purchase Price	Quantity
# GOOG	7/5/2018	281.79	10
# upload in new portfolio https://finance.yahoo.com/portfolio/p_1/view/v1
# yahoo error 10/16, no apparent issue

win_outpath = "/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/excel_pipeline/current"
import pandas as pd
csvfile = f"{win_outpath}/acct_summary_by_symbol.csv"
acct_summary_df = pd.read_csv(csvfile)
acct_summary_df.head()


acct_summary_df.rename(columns={"symbol": "Symbol", "current_price": "Purchase Price", "total_quantity": "Quantity"}, inplace=True)
acct_summary_df["Trade Date"] = today_str
acct_summary_df.drop(columns=["avg_price_paid", "accounts", "total_value"], inplace=True)
# reorder columns for Yahoo format
acct_summary_df = acct_summary_df[["Symbol", "Trade Date", "Purchase Price", "Quantity"]]
acct_summary_df.to_csv(f"{win_outpath}/acct_summary_by_symbol_yahoo.csv", index=False)
acct_summary_df.head()


,Symbol,Trade Date,Purchase Price,Quantity
0,FSTA,2026-11-02,56.48,370
1,GLD,2026-11-02,355.78,106
2,HODL,2026-11-02,21.66,28
3,IDHQ,2026-11-02,44.03,700
4,IVW,2026-11-02,134.46,230
